In [138]:
import requests

from bs4 import BeautifulSoup


BASE_URL = "https://sinca.mma.gob.cl/index.php/region/index/id/"
REGIONS = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XI", "XII", "XIII", "XIV", "XV", "XVI", "M"]

# Hit the base URL and try to extract all station IDs and their parameters
# If not possible, extract all station IDs and iterate thourgh them to take the parameters
# Store them in a json file within the data directory
# This job should be done once every 6 months or after a site change is made 

In [ ]:
import re

from urllib.parse import urlparse, parse_qs


rows = []

for region in REGIONS:
    resp = requests.get(BASE_URL+region)
    soup = BeautifulSoup(resp.text, "html.parser")

    for a in soup.find_all("a", href=re.compile(r"macropath=")):
        qs = parse_qs(urlparse(str(a["href"])).query)
        parts = qs.get("macropath", [""])[0].strip("./").split("/")
        if len(parts) != 4:
            continue
        region, station, kind, param = parts
        rows.append({"region": region, "station": station, "kind": kind, "param": param,
                    "name": qs.get("header", [""])[0],
                    "from": qs.get("from", [""])[0], "to": qs.get("to", [""])[0]})


In [156]:
import pandas as pd

from datetime import date
from pathlib import Path


Path("data/catalogs").mkdir(parents=True, exist_ok=True)
today = date.today().strftime("%y%m%d")

df = pd.DataFrame(rows)
df.sort_values(["region","station","kind","param"]).to_csv(f"data/catalogs/{today}.csv", index=False)